# Hello World

`tests/unit/test_hello_world.py`와 같은 흐름으로 `catcher_llm.services.helloworld`를 실행합니다.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "catcher_llm").exists():
            return candidate
    raise RuntimeError("Unable to locate project root from the current working directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"


def main() -> int:
    os.environ.setdefault("LANGSMITH_TRACING", "false")

    if sys.version_info < (3, 12) and VENV_PYTHON.exists():
        env = os.environ.copy()
        code = (
            "import os\n"
            "import sys\n"
            f"sys.path.insert(0, {str(SRC_DIR)!r})\n"
            "os.environ.setdefault('LANGSMITH_TRACING', 'false')\n"
            "from catcher_llm.services.helloworld import build_hello_world_chain\n"
            "print(build_hello_world_chain())\n"
        )
        completed = subprocess.run(
            [str(VENV_PYTHON), "-c", code],
            check=False,
            cwd=PROJECT_ROOT,
            env=env,
            capture_output=True,
            text=True,
        )
        if completed.stdout:
            print(completed.stdout, end="")
        if completed.stderr:
            print(completed.stderr, end="", file=sys.stderr)
        return completed.returncode

    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

    from catcher_llm.services.helloworld import build_hello_world_chain

    result = build_hello_world_chain()
    print(result)
    return 0


main()